# PhoBERT Internal Fine-tune V3
**Advanced**: FGM Adversarial Training + Label Smoothing + Sarcasm Augmentation
**Target**: >95% accuracy

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
import torch, numpy as np, pandas as pd, os, json, warnings
import matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
warnings.filterwarnings('ignore')
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import torch.nn as nn, torch.nn.functional as F

SEED=42; torch.manual_seed(SEED); np.random.seed(SEED)
LABEL_NAMES={0:'NEGATIVE',1:'POSITIVE',2:'NEUTRAL'}
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PUBLIC_MODEL='/content/drive/MyDrive/DO_AN_1_main/phobert-sentiment-3class'
OUT_DIR='/content/drive/MyDrive/DO_AN_1_main/phobert-sentiment-3class-internal-v3'
DATA_DIR='/content/drive/MyDrive/DO_AN_1_main/data'
os.makedirs(OUT_DIR, exist_ok=True)
print('Public model exists:',os.path.exists(PUBLIC_MODEL))

## Load Data + Sarcasm Augmentation (x8 repeat)

In [ ]:
df=pd.concat([pd.read_csv(f'{DATA_DIR}/{f}') for f in ['negative_content.csv','positive_content.csv','neutral_content.csv']],ignore_index=True)
df=df.dropna(subset=['text']); df=df[df['text'].str.strip().str.len()>0]
df['label']=df['label'].astype(int); df=df[['text','label']].reset_index(drop=True)
print(f'Internal: {len(df):,}')

In [ ]:
# Comprehensive sarcasm + hard examples
SARC_NEG=[
'Tuyet voi lun, lan sau khong bao gio mua nua','10 diem cho su that vong',
'Qua tuyet voi, tuyet voi den muc khong the dung duoc','Cam on shop, lan sau khong bao gio quay lai',
'Dep lam, dep nhu hang cho','Chat luong 5 sao, 5 sao cho su that vong',
'Hang xin qua, xin den muc hu ngay khi mo hop','Tuyet voi, dat tien ma nhan ve do rac',
'OK lun, ok la khong bao gio mua lai','Xuat sac, xuat sac trong viec lam khach that vong',
'10/10 cho su te hai','Tot lam, tot den muc phai di bao hanh ngay',
'Ung ho shop, ung ho bang cach khong quay lai','5 sao vi khong co 0 sao',
'Rat tot, tot cho nhung ai muon mat tien','Hang dep lam luon, dep den muc khong dam xai',
'Shop uy tin lam, giao hang sai het','Dich vu tot qua, doi ca tuan khong thay tra loi',
'Cam on da cho trai nghiem toi te nhat','Tuyet voi that su, se khong gioi thieu cho ai',
'Gia tot, tot cho shop thoi chu khong phai cho khach','Giao hang nhanh that, nhanh den muc hang bi vo',
'Dong goi can than lam, can than den muc hang bi bep','Mua 10 cai hu 9, shop lam an tot that',
'Qua hay, hay den muc muon khoc','Dang dong tien that, dong tien bo di',
'Rat an tuong, an tuong vi su te hai','Oki lun, se review xau cho shop',
'Mong doi nhieu, that vong con nhieu hon','Nhin hinh thi dep, nhan hang thi muon khoc',
'Mo ta mot dang, thuc te mot neo','Tien mat tat mang',
'Lan dau va cung la lan cuoi mua o day','Gia re co ly do cua no',
'Duoc cai vo dep, con lai thi thoi','Khong biet khen gi, vi khong co gi de khen',
'That su khong the nao te hon duoc nua','Cam on shop da cho bai hoc nho doi',
'Hang nay ma 5 sao thi troi oi','Dep ghia, dep kieu lua dao',
'Wow qua tot luon, tot den muc phai tra lai','Se gioi thieu cho ke thu mua',
]
HARD_NEG=[
'Chat luong kem','Kem qua','Te lam','Qua te','Khong tot','Khong dep',
'Khong dang tien','Khong nhu mong doi','That vong','Qua that vong',
'Chan lam','Do rac','Phi tien','Mat tien','Gia ma','Hang gia',
'Khong bao gio mua lai','Te nhat tung thay','Tham hoa','Dung mua',
'Lua dao','Sai mau','Sai size','Hang loi','Hu roi',
'Khong su dung duoc','Hong ngay lan dau','Kem chat luong',
'Hang kem chat luong','Qua kem','San pham te','Hang do','Hang rac',
]
HARD_POS=[
'Khong co gi de che','Khong co gi phan nan','Khong that vong',
'Dat nhung xung dang','Chua thay loi gi','Vuot mong doi',
'Dung chat luong','Xai tot','Ban dep','Mua lan 3 roi',
'Se gioi thieu ban be','Dung nhu quang cao','Chat luong that su tot',
'Rat dang tien','Hang dep dung nhu mo ta','Tren ca tuyet voi',
]
HARD_NEU=[
'Giao hang nhanh','Da nhan hang','Hang ve roi','Dong goi ok',
'Ship dung hen','Nhan duoc roi','Chua dung nen chua biet',
'Se danh gia sau','Moi nhan chua mo','Binh thuong','Tam on',
'Cung duoc','Khong tot khong xau','Trung binh','Chua test',
]

AUG=8  # Repeat 8x for strong signal
aug=[]
for _ in range(AUG):
    aug+=[{'text':t,'label':0} for t in SARC_NEG+HARD_NEG]
    aug+=[{'text':t,'label':1} for t in HARD_POS]
    aug+=[{'text':t,'label':2} for t in HARD_NEU]
df_aug=pd.DataFrame(aug)
df_all=pd.concat([df,df_aug],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
print(f'After augmentation: {len(df_all):,} (added {len(df_aug)} aug samples)')

## Split + Weights + Tokenize

In [ ]:
df_train,df_temp=train_test_split(df_all,test_size=0.2,random_state=SEED,stratify=df_all['label'])
df_val,df_test=train_test_split(df_temp,test_size=0.5,random_state=SEED,stratify=df_temp['label'])
print(f'Train:{len(df_train):,} Val:{len(df_val):,} Test:{len(df_test):,}')

cw=compute_class_weight('balanced',classes=np.array([0,1,2]),y=np.array(df_train['label']))
cw_tensor=torch.tensor(cw,dtype=torch.float32).to(device)
print('Weights:',{LABEL_NAMES[i]:f'{w:.3f}' for i,w in enumerate(cw)})

In [ ]:
MAX_LEN=128
tokenizer=AutoTokenizer.from_pretrained(PUBLIC_MODEL)
model=AutoModelForSequenceClassification.from_pretrained(PUBLIC_MODEL,num_labels=3).to(device)
print(f'Model loaded: {model.num_parameters():,} params')

def tok_fn(ex): return tokenizer(ex['text'],padding='max_length',truncation=True,max_length=MAX_LEN,return_tensors=None)
tk_train=Dataset.from_pandas(df_train[['text','label']].reset_index(drop=True)).map(tok_fn,batched=True)
tk_val=Dataset.from_pandas(df_val[['text','label']].reset_index(drop=True)).map(tok_fn,batched=True)
tk_test=Dataset.from_pandas(df_test[['text','label']].reset_index(drop=True)).map(tok_fn,batched=True)
for d in [tk_train,tk_val,tk_test]: d.set_format('torch',columns=['input_ids','attention_mask','label'])
print(f'Tokenized: train={len(tk_train)} val={len(tk_val)} test={len(tk_test)}')

## FGM Adversarial + Focal Loss + Label Smoothing

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self,alpha=None,gamma=2.5,label_smoothing=0.1):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma; self.ls=label_smoothing
    def forward(self,logits,targets):
        ce=F.cross_entropy(logits,targets,weight=self.alpha,reduction='none',label_smoothing=self.ls)
        pt=F.softmax(logits,dim=1).gather(1,targets.unsqueeze(1)).squeeze(1)
        return ((1-pt)**self.gamma * ce).mean()

class V3Trainer(Trainer):
    """FGM Adversarial Training + Focal Loss + Label Smoothing"""
    def __init__(self,*a,class_weights=None,focal_gamma=2.5,label_smoothing=0.1,fgm_eps=0.5,**kw):
        super().__init__(*a,**kw)
        self.cw=class_weights; self.fg=focal_gamma; self.ls=label_smoothing
        self.fgm_eps=fgm_eps; self._fl=None; self._bk={}

    def _loss(self,model,inputs):
        labels=inputs['labels']
        out=model(input_ids=inputs['input_ids'],attention_mask=inputs['attention_mask'])
        if self._fl is None:
            w=self.cw.to(out.logits.device) if self.cw is not None else None
            self._fl=FocalLoss(alpha=w,gamma=self.fg,label_smoothing=self.ls)
        return self._fl(out.logits,labels),out

    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        loss,out=self._loss(model,inputs)
        return (loss,out) if return_outputs else loss

    def training_step(self,model,inputs,num_items_in_batch=None):
        model.train()
        inputs=self._prepare_inputs(inputs)
        # Normal forward+backward
        loss,_=self._loss(model,inputs)
        if self.args.gradient_accumulation_steps>1:
            loss=loss/self.args.gradient_accumulation_steps
        self.accelerator.backward(loss)
        # FGM: perturb word embeddings
        for n,p in model.named_parameters():
            if p.requires_grad and 'word_embeddings' in n and p.grad is not None:
                self._bk[n]=p.data.clone()
                norm=torch.norm(p.grad)
                if norm!=0 and not torch.isnan(norm):
                    p.data.add_(self.fgm_eps*p.grad/norm)
        # Adversarial forward+backward
        loss_adv,_=self._loss(model,inputs)
        if self.args.gradient_accumulation_steps>1:
            loss_adv=loss_adv/self.args.gradient_accumulation_steps
        self.accelerator.backward(loss_adv)
        # Restore
        for n,p in model.named_parameters():
            if n in self._bk: p.data=self._bk[n]
        self._bk={}
        return loss.detach()

def compute_metrics(ep):
    logits,labels=ep
    preds=np.argmax(logits,axis=-1)
    acc=accuracy_score(labels,preds)
    p,r,f1,_=precision_recall_fscore_support(labels,preds,average='macro',zero_division=0)
    return {'accuracy':acc,'precision':p,'recall':r,'f1':f1}

print('V3 Trainer ready: FGM + Focal Loss (gamma=2.5) + Label Smoothing (0.1)')

## Training

In [ ]:
args=TrainingArguments(
    output_dir='./results_v3',
    num_train_epochs=8, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, gradient_accumulation_steps=4,
    learning_rate=5e-6, weight_decay=0.01,
    warmup_ratio=0.1, lr_scheduler_type='cosine',
    eval_strategy='steps', eval_steps=200,
    logging_steps=100, save_strategy='steps',
    save_steps=200, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model='f1',
    greater_is_better=True, fp16=True,
    dataloader_num_workers=2, seed=SEED, report_to='none',
    max_grad_norm=1.0,
)
trainer=V3Trainer(
    model=model, args=args, train_dataset=tk_train, eval_dataset=tk_val,
    compute_metrics=compute_metrics, class_weights=cw_tensor,
    focal_gamma=2.5, label_smoothing=0.1, fgm_eps=0.5,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)
print(f'LR=5e-6 | Epochs=8 | EffBatch=64 | FGM=0.5 | Gamma=2.5 | LS=0.1')

In [ ]:
print('='*60+'\nSTARTING V3 TRAINING\n'+'='*60)
train_result=trainer.train()
print(f'\nDone: {train_result.metrics["train_runtime"]:.0f}s')

## Evaluation

In [ ]:
print('='*60+'\nTEST SET EVALUATION\n'+'='*60)
ev=trainer.evaluate(tk_test)
print(f'Accuracy:  {ev["eval_accuracy"]*100:.2f}%')
print(f'Precision: {ev["eval_precision"]*100:.2f}%')
print(f'Recall:    {ev["eval_recall"]*100:.2f}%')
print(f'F1 macro:  {ev["eval_f1"]*100:.2f}%')

In [ ]:
po=trainer.predict(tk_test)
preds=np.argmax(po.predictions,axis=-1); true=po.label_ids
cm=confusion_matrix(true,preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
  xticklabels=['NEG','POS','NEU'],yticklabels=['NEG','POS','NEU'])
plt.title('V3 Confusion Matrix',fontsize=14,fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix_v3.png',dpi=150); plt.show()
print('\n'+classification_report(true,preds,target_names=['NEG','POS','NEU'],digits=4))

In [ ]:
def predict(text):
    model.eval()
    inp=tokenizer(text,padding='max_length',truncation=True,max_length=MAX_LEN,return_tensors='pt')
    inp={k:v.to(device) for k,v in inp.items()}
    with torch.no_grad():
        pr=F.softmax(model(**inp).logits,dim=-1)
        c=torch.argmax(pr,dim=-1).item()
    return c,pr[0][c].item()

TEST=[
('San pham rat tot, chat luong tuyet voi',1,'POS'),('Hang te qua, khong dung mo ta',0,'NEG'),
('Da nhan hang, chua mo',2,'NEU'),('Shop lua dao, mat tien oan',0,'NEG'),
('5 sao, se ung ho tiep',1,'POS'),('OK',2,'NEU'),('Tam duoc',2,'NEU'),
('Gia re ma tot',1,'POS'),('Chat luong kem',0,'NEG'),('Giao hang nhanh',2,'NEU'),
('Hang dep nhung giao cham',1,'POS'),('Gia dat nhung chat luong tot',1,'POS'),
('Ship nhanh, dong goi can than',2,'NEU'),('Mau khong giong hinh lam',0,'NEG'),
('Chua dung nen chua biet',2,'NEU'),('Tuyet voi lun, lan sau khong mua nua',0,'NEG'),
('10 diem cho su that vong',0,'NEG'),('Mua nhieu lan roi, van tin tuong shop',1,'POS'),
('Khong co gi de che',1,'POS'),('Binh thuong, khong dac biet',2,'NEU'),
]
print('='*80+'\n20 SENTENCES TEST (V3)\n'+'='*80)
ok=0; res=[]
for i,(t,el,en) in enumerate(TEST,1):
    pl,cf=predict(t); pn=LABEL_NAMES[pl]; c=pl==el
    if c: ok+=1
    s='[OK]' if c else '[X]'
    res.append({'No':i,'Text':t,'Exp':en,'Pred':pn,'Conf':cf,'OK':c})
    print(f'[{i:02d}] {t}\n     {en} -> {pn} ({cf*100:.1f}%) {s}')
a20=ok/len(TEST)*100
print(f'\nResult: {ok}/{len(TEST)} = {a20:.1f}%')
print(f'V1=75.0% | V2=80.0% | V3={a20:.1f}% ({a20-80:+.1f}%)')

## Save Model

In [ ]:
model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
cfg={'version':'V3','techniques':['FGM_adversarial','focal_loss_gamma2.5','label_smoothing_0.1','cosine_schedule','sarcasm_augment_x8'],'lr':5e-6,'epochs':8,'fgm_epsilon':0.5,'eval':{'acc':ev['eval_accuracy'],'f1':ev['eval_f1'],'sent_test':a20}}
with open(f'{OUT_DIR}/config.json','w') as f: json.dump(cfg,f,indent=2)
print(f'Saved to {OUT_DIR}')
for fn in os.listdir(OUT_DIR):
    sz=os.path.getsize(os.path.join(OUT_DIR,fn))/1024/1024
    print(f'  {fn}: {sz:.2f}MB')

In [ ]:
# Verify
print('Verifying saved model...')
m2=AutoModelForSequenceClassification.from_pretrained(OUT_DIR).to(device).eval()
t2=AutoTokenizer.from_pretrained(OUT_DIR)
for t in ['San pham rat tot','Chat luong kem','Binh thuong','Tuyet voi lun, khong mua nua','10 diem cho su that vong','Giao hang nhanh']:
    inp=t2(t,padding='max_length',truncation=True,max_length=MAX_LEN,return_tensors='pt')
    inp={k:v.to(device) for k,v in inp.items()}
    with torch.no_grad():
        pr=F.softmax(m2(**inp).logits,dim=-1)
        c=torch.argmax(pr,dim=-1).item()
    print(f"  '{t}' -> {LABEL_NAMES[c]} ({pr[0][c].item()*100:.1f}%)")
print('Done!')